In [ ]:
"""
pathok_common.py
================
Shared machinery for the Tier-1 experiments. Import this first, then run any
of exp1..exp5. On Kaggle: paste this whole file into one cell and run it, then
paste the experiment file into the next cell.

    %%writefile pathok_common.py
    ...            # or just paste and exec

Everything here is lifted from your existing scripts with three changes:

 1. reload_split() no longer requires a checkpoint. w2i can be supplied, taken
    from a checkpoint, or rebuilt with sorted(). All experiments must use the
    SAME w2i, so pass it explicitly once you have it.

 2. page_of() is added. BN-HTRd paths are <ROOT>/<writer>/Words/<page>/<word>.jpg,
    so page id is recoverable. Probe train/test splits are made BY PAGE inside
    each writer, otherwise two crops from the same line share ink thickness,
    lighting and pen pressure and the probe reports memorisation of the page,
    not of the writer.

 3. Cluster bootstrap over WRITERS. Every headline number in the paper is an
    average over 20-74 writers, not over 1500 independent words, so word-level
    CIs understate the uncertainty by roughly sqrt(words_per_writer).

NOTHING here trains anything. It is safe to import repeatedly.
"""

import os
import glob
import json
import copy
import random
import warnings

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import editdistance
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import VisionEncoderDecoderModel, AutoImageProcessor
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve

warnings.filterwarnings("ignore")


# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════
class S:
    # ---- EDIT THESE ---------------------------------------------------------
    CKPT_ROOT = "/kaggle/input/notebooks/kayed007/fork-of-retrain-pathok/pathok_clean"
    ROOT      = "/kaggle/input/datasets/kaidul007/bnhtrd/Dataset"
    # -------------------------------------------------------------------------
    TROCR_MODEL = "microsoft/trocr-base-handwritten"
    CKPT_FILE   = "baseline_clean_final.pt"
    SPLIT_FILE  = "split.json"
    VOCAB_FILE  = "vocab_clean.json"

    SEED = 42
    VAL_FRAC = 0.05          # must match the original retrain
    BATCH = 16
    NUM_WORKERS = 2
    TARGET_H = 64
    TARGET_W = 256

    CACHE = "/kaggle/working/df_cache.parquet"


PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3
SPECIAL_TOKENS = 4
MAX_LEN = 64
PUNCT = {'.', ':', '।', ',', ';', '!', '?', '-', '–', '—'}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def seed_all(seed=S.SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_all()


# ══════════════════════════════════════════════════════════════════════════════
# GRAPHEMES / VOCAB
# ══════════════════════════════════════════════════════════════════════════════
HASANTA, CHANDRABINDU, ANUSVARA = '\u09CD', '\u0981', '\u0982'
VISARGA, NUKTA = '\u0983', '\u09BC'
VOWEL_SIGNS = set('\u09BE\u09BF\u09C0\u09C1\u09C2\u09C3\u09C4'
                  '\u09C7\u09C8\u09CB\u09CC\u09D7')
INDEPENDENT_VOWELS = set('\u0985\u0986\u0987\u0988\u0989\u098A\u098B'
                         '\u098F\u0990\u0993\u0994')
CONSONANTS = set(chr(c) for c in range(0x0995, 0x09BA)) | {
    '\u09DC', '\u09DD', '\u09DF', '\u09CE', '\u09B0'}


def segment_graphemes(word):
    out, i, n = [], 0, len(word)
    while i < n:
        ch = word[i]
        if ch in INDEPENDENT_VOWELS:
            g = ch; i += 1
            while i < n and word[i] in {CHANDRABINDU, ANUSVARA, VISARGA}:
                g += word[i]; i += 1
            out.append(g)
        elif ch in CONSONANTS:
            g = ch; i += 1
            if i < n and word[i] == NUKTA:
                g += word[i]; i += 1
            while i + 1 < n and word[i] == HASANTA and word[i + 1] in CONSONANTS:
                g += word[i] + word[i + 1]; i += 2
            if i < n and word[i] in VOWEL_SIGNS:
                g += word[i]; i += 1
            out.append(g)
        else:
            out.append(ch); i += 1
    return out


def is_bangla_word(t):
    c = [x for x in t if not x.isspace()]
    return bool(c) and sum(1 for x in c if '\u0980' <= x <= '\u09FF') / len(c) > 0.8


class GraphemeVocab:
    def __init__(self):
        self.g2i = {}
        self.i2g = {PAD_ID: "<pad>", UNK_ID: "<unk>", BOS_ID: "<s>", EOS_ID: "</s>"}
        self.vocab_size = SPECIAL_TOKENS

    def build(self, labels):
        gs = sorted({g for w in labels for g in segment_graphemes(str(w))})
        for i, g in enumerate(gs):
            t = i + SPECIAL_TOKENS
            self.g2i[g] = t; self.i2g[t] = g
        self.vocab_size = SPECIAL_TOKENS + len(gs)
        return self

    def encode(self, w):
        return [self.g2i.get(g, UNK_ID) for g in segment_graphemes(str(w))]

    def decode(self, ids):
        out = []
        for t in ids:
            t = int(t)
            if t == EOS_ID:
                break
            if t in (PAD_ID, BOS_ID, UNK_ID):
                continue
            out.append(self.i2g.get(t, ""))
        return "".join(out)

    def save(self, p):
        json.dump({"g2i": self.g2i,
                   "i2g": {str(k): v for k, v in self.i2g.items()},
                   "vocab_size": self.vocab_size},
                  open(p, "w", encoding="utf-8"), ensure_ascii=False)

    @classmethod
    def load(cls, p):
        d = json.load(open(p, encoding="utf-8"))
        v = cls(); v.g2i = d["g2i"]
        v.i2g = {int(k): x for k, x in d["i2g"].items()}
        v.vocab_size = d["vocab_size"]
        return v


def encode_label(vocab, text):
    seq = (vocab.encode(text) + [EOS_ID])[:MAX_LEN]
    return seq + [PAD_ID] * (MAX_LEN - len(seq))


# ══════════════════════════════════════════════════════════════════════════════
# PREPROCESSING  (height + contrast, squeeze not truncate)
# ══════════════════════════════════════════════════════════════════════════════
def preprocess(gray, cfg=S):
    h, w = gray.shape
    scale = cfg.TARGET_H / max(h, 1)
    new_w = max(1, int(round(w * scale)))
    if new_w > cfg.TARGET_W:
        content = cv2.resize(gray, (cfg.TARGET_W, cfg.TARGET_H),
                             interpolation=cv2.INTER_AREA)
    else:
        content = cv2.resize(gray, (new_w, cfg.TARGET_H),
                             interpolation=cv2.INTER_LINEAR)
    c = content.astype(np.float32)
    c = (c - c.mean()) / (c.std() + 1e-6)
    content = np.clip(c * 48 + 200, 0, 255).astype(np.uint8)
    canvas = np.full((cfg.TARGET_H, cfg.TARGET_W), 255, dtype=np.uint8)
    canvas[:, :content.shape[1]] = content
    return canvas


class WDataset(Dataset):
    def __init__(self, df, vocab, fe, cfg=S):
        self.df = df.reset_index(drop=True)
        self.vocab, self.fe, self.cfg = vocab, fe, cfg

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        gray = cv2.imread(r["image_path"], cv2.IMREAD_GRAYSCALE)
        if gray is None:
            gray = np.full((self.cfg.TARGET_H, self.cfg.TARGET_W), 255, np.uint8)
        img = Image.fromarray(cv2.cvtColor(preprocess(gray, self.cfg),
                                           cv2.COLOR_GRAY2RGB))
        pv = self.fe(img, return_tensors="pt").pixel_values.squeeze(0)
        lb = torch.tensor(encode_label(self.vocab, str(r["label"])), dtype=torch.long)
        lb[lb == PAD_ID] = -100
        return {"pixel_values": pv, "labels": lb,
                "wid": torch.tensor(int(r["wid"]), dtype=torch.long)}


# ══════════════════════════════════════════════════════════════════════════════
# DATA / SPLIT
# ══════════════════════════════════════════════════════════════════════════════
def writer_of(p, root):
    return os.path.relpath(p, root).split(os.sep)[0]


def page_of(p, root):
    """<ROOT>/<writer>/Words/<page>/<word>.jpg -> '<writer>/<page>'.
    Probe splits go by page: two crops from the same line share ink weight,
    lighting and pen pressure, so a within-page split measures page recall."""
    parts = os.path.relpath(p, root).split(os.sep)
    return f"{parts[0]}/{parts[2]}" if len(parts) >= 4 else parts[0]


def load_with_writers(cfg=S, use_cache=True):
    if use_cache and cfg.CACHE and os.path.exists(cfg.CACHE):
        df = pd.read_parquet(cfg.CACHE)
        print(f"  [cache] {len(df):,} samples | writers {df['writer'].nunique()}")
        return df

    label_dfs = []
    for f in glob.glob(cfg.ROOT + "/*/*.xlsx"):
        try:
            for d in pd.read_excel(f, sheet_name=None, engine="openpyxl").values():
                if d.shape[1] < 2:
                    continue
                d = d.iloc[:, :2].copy(); d.columns = ["Id", "Word"]
                label_dfs.append(d)
        except Exception:
            continue
    if not label_dfs:
        raise FileNotFoundError(f"No .xlsx under {cfg.ROOT}/*/*.xlsx -- check S.ROOT")
    labels_df = (pd.concat(label_dfs, ignore_index=True)
                 .drop_duplicates(subset="Id", keep="first"))
    imgs = (glob.glob(cfg.ROOT + "/*/Words/*/*.jpg")
            + glob.glob(cfg.ROOT + "/*/Words/*/*.JPG"))
    image_df = pd.DataFrame({"Id": [os.path.basename(p).split(".")[0] for p in imgs],
                             "image_path": imgs})
    df = (image_df.merge(labels_df, on="Id", how="inner")
          .rename(columns={"Word": "label"})[["image_path", "label"]].dropna()
          .reset_index(drop=True))
    df["label"] = df["label"].astype(str).str.strip()
    df = df[(df["label"] != "") & (~df["label"].isin(PUNCT))]
    df = df[df["label"].apply(is_bangla_word)]
    df = df[df["image_path"].apply(os.path.exists)].reset_index(drop=True)
    df["writer"] = df["image_path"].apply(lambda p: writer_of(p, cfg.ROOT))
    df["page"] = df["image_path"].apply(lambda p: page_of(p, cfg.ROOT))
    print(f"  samples: {len(df):,} | writers: {df['writer'].nunique()} "
          f"| pages: {df['page'].nunique()}")
    if cfg.CACHE:
        try:
            df.to_parquet(cfg.CACHE)
        except Exception as e:
            print(f"  [cache] skipped: {e}")
    return df


def reload_split(df, cfg=S, w2i=None, ck=None, verbose=True):
    """Rebuild the frozen split. Pass w2i explicitly (preferred) so every
    experiment agrees on writer->index. Falls back to the checkpoint, then to
    sorted() -- note string sort puts '10' before '2'."""
    spec = json.load(open(os.path.join(cfg.CKPT_ROOT, cfg.SPLIT_FILE)))
    test_w, forget_w, retain_w = (set(map(str, spec[k]))
                                  for k in ("test", "forget", "retain"))
    if w2i is None:
        if ck is not None and "w2i" in ck:
            w2i = {str(k): int(v) for k, v in ck["w2i"].items()}
        else:
            w2i = {w: i for i, w in enumerate(sorted(retain_w | forget_w))}

    test_df = df[df["writer"].isin(test_w)].reset_index(drop=True)
    forget_df = df[df["writer"].isin(forget_w)].reset_index(drop=True)
    retain_all = df[df["writer"].isin(retain_w)].reset_index(drop=True)
    retain_df, val_df = train_test_split(retain_all, test_size=cfg.VAL_FRAC,
                                         random_state=cfg.SEED)
    retain_df = retain_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    for d in (retain_df, forget_df, val_df):
        d["wid"] = d["writer"].map(w2i).fillna(-1).astype(int)
    test_df["wid"] = -1

    assert not (set(test_df["writer"]) & (set(retain_df["writer"])
                                          | set(forget_df["writer"]))), "TEST LEAK"
    if verbose:
        print(f"  retain {len(retain_df):,}/{retain_df['writer'].nunique()}w (MEMBERS) | "
              f"forget {len(forget_df):,}/{forget_df['writer'].nunique()}w | "
              f"val {len(val_df):,} | "
              f"test {len(test_df):,}/{test_df['writer'].nunique()}w (NON-MEMBERS)")
    return retain_df, forget_df, val_df, test_df, w2i


def sample_per_writer(df, k, seed=S.SEED):
    return (df.groupby("writer", group_keys=False)
              .apply(lambda g: g.sample(min(k, len(g)), random_state=seed))
              .reset_index(drop=True))


# ══════════════════════════════════════════════════════════════════════════════
# MODEL
# ══════════════════════════════════════════════════════════════════════════════
class MultiHead(nn.Module):
    def __init__(self, ved, n_writers):
        super().__init__()
        self.ved = ved
        self.writer_head = nn.Linear(ved.config.encoder.hidden_size, n_writers)
        self.n_writers = n_writers

    def forward(self, pv, labels=None):
        enc = self.ved.encoder(pixel_values=pv, return_dict=True)
        wlog = self.writer_head(enc.last_hidden_state[:, 0, :])
        rec = (self.ved(encoder_outputs=enc, labels=labels, return_dict=True)
               if labels is not None else None)
        return rec, wlog


def build_model(vocab, n_writers, cfg=S):
    ved = VisionEncoderDecoderModel.from_pretrained(cfg.TROCR_MODEL)
    ved.decoder.resize_token_embeddings(vocab.vocab_size)
    ved.config.vocab_size = vocab.vocab_size
    ved.config.decoder.vocab_size = vocab.vocab_size
    for c in (ved.config, ved.generation_config):
        c.decoder_start_token_id = BOS_ID; c.pad_token_id = PAD_ID
        c.eos_token_id = EOS_ID; c.bos_token_id = BOS_ID
    ved.generation_config.max_new_tokens = MAX_LEN
    return MultiHead(ved, n_writers)


def load_into(model, sd, label="", strict_report=True):
    res = model.load_state_dict(sd, strict=False)
    crit = [k for k in res.missing_keys if not k.startswith("ved.encoder.pooler")]
    if crit:
        raise RuntimeError(f"[{label}] {len(crit)} critical keys missing, "
                           f"e.g. {crit[:5]} -- every number would be fiction")
    if strict_report and res.unexpected_keys:
        print(f"  [{label}] unexpected keys: {list(res.unexpected_keys)[:5]}")
    return model.to(DEVICE).eval()


def restore(model, state):
    model.load_state_dict(state, strict=True)
    return model


def get_feature_extractor(cfg=S):
    return AutoImageProcessor.from_pretrained(cfg.TROCR_MODEL)


# ══════════════════════════════════════════════════════════════════════════════
# EVALUATION
# ══════════════════════════════════════════════════════════════════════════════
@torch.no_grad()
def writer_acc(model, df, vocab, fe, cfg=S, n=800, return_per_writer=False):
    d = (df if n is None else
         df.sample(min(n, len(df)), random_state=cfg.SEED)).reset_index(drop=True)
    loader = DataLoader(WDataset(d, vocab, fe, cfg), batch_size=cfg.BATCH,
                        shuffle=False, num_workers=cfg.NUM_WORKERS)
    was = model.training; model.eval()
    hits = []
    for b in loader:
        pv = b["pixel_values"].to(DEVICE); wid = b["wid"].to(DEVICE)
        enc = model.ved.encoder(pixel_values=pv, return_dict=True)
        wl = model.writer_head(enc.last_hidden_state[:, 0, :])
        hits.append((wl.argmax(-1) == wid).float().cpu())
    if was:
        model.train()
    hits = torch.cat(hits).numpy() if hits else np.array([])
    acc = 100.0 * hits.mean() if len(hits) else float("nan")
    if return_per_writer:
        return acc, pd.DataFrame({"writer": d["writer"].values, "hit": hits})
    return acc


@torch.no_grad()
def rec_quality(model, df, vocab, fe, cfg=S, n=300, return_per_word=False):
    d = (df if n is None else
         df.sample(min(n, len(df)), random_state=cfg.SEED)).reset_index(drop=True)
    loader = DataLoader(WDataset(d, vocab, fe, cfg), batch_size=cfg.BATCH,
                        shuffle=False, num_workers=cfg.NUM_WORKERS)
    was = model.training; model.eval(); preds = []
    for b in loader:
        gen = model.ved.generate(b["pixel_values"].to(DEVICE),
                                 max_new_tokens=MAX_LEN, num_beams=1)
        preds += [vocab.decode(r) for r in gen.cpu().tolist()]
    if was:
        model.train()
    gts = d["label"].astype(str).tolist()
    dist = np.array([editdistance.eval(list(p), list(g)) for p, g in zip(preds, gts)])
    gl = np.array([max(len(g), 1) for g in gts])
    keep = np.array([len(p) for p in preds]) <= 3 * gl + 5
    out = {"WER": 100 * np.mean([p != g for p, g in zip(preds, gts)]),
           "CER": 100 * dist[keep].sum() / gl[keep].sum() if keep.any() else float("nan"),
           "runaway": int((~keep).sum())}
    if return_per_word:
        out["per_word"] = pd.DataFrame({"writer": d["writer"].values,
                                        "dist": dist, "gtlen": gl,
                                        "wrong": [p != g for p, g in zip(preds, gts)]})
    return out


@torch.no_grad()
def writer_rank_diag(model, df, vocab, fe, n_classes, cfg=S, n=400):
    """Rank of the TRUE writer's logit among all classes, 1-indexed.
    Interpret against the ORACLE's rank, never against n_classes/2 -- a model
    retrained without these writers also ranks them low, so 'inversion' is only
    meaningful as a deviation from what exact deletion produces."""
    d = df.sample(min(n, len(df)), random_state=cfg.SEED).reset_index(drop=True)
    loader = DataLoader(WDataset(d, vocab, fe, cfg), batch_size=cfg.BATCH,
                        shuffle=False, num_workers=cfg.NUM_WORKERS)
    model.eval(); ranks, acc, writers = [], [], []
    for b in loader:
        pv = b["pixel_values"].to(DEVICE); wid = b["wid"].to(DEVICE)
        enc = model.ved.encoder(pixel_values=pv, return_dict=True)
        wl = model.writer_head(enc.last_hidden_state[:, 0, :]).float()
        true = wl.gather(1, wid[:, None])
        ranks.append(((wl > true).sum(1) + 1).float().cpu())
        acc.append((wl.argmax(-1) == wid).float().cpu())
    r = torch.cat(ranks).numpy()
    return {"mean_rank": float(r.mean()), "median_rank": float(np.median(r)),
            "frac_last": float((r == n_classes).mean()),
            "ACC": 100 * float(torch.cat(acc).mean()),
            "ranks": r, "writer": d["writer"].values}


# ══════════════════════════════════════════════════════════════════════════════
# MIA FEATURES  (12-dim teacher-forced decoder confidence, no writer head)
# ══════════════════════════════════════════════════════════════════════════════
FEATNAMES = ["mean_lp", "min_lp", "max_lp", "std_lp", "sum_lp", "mean_ent",
             "mean_margin", "mean_top1p", "frac_correct", "mean_logrank",
             "n_tokens", "last_lp"]


@torch.no_grad()
def extract_features(model, df, vocab, fe, cfg=S):
    loader = DataLoader(WDataset(df, vocab, fe, cfg), batch_size=cfg.BATCH,
                        shuffle=False, num_workers=cfg.NUM_WORKERS)
    model.eval(); feats = []
    for b in loader:
        pv = b["pixel_values"].to(DEVICE); lab = b["labels"].to(DEVICE)
        logits = model.ved(pixel_values=pv, labels=lab, return_dict=True).logits.float()
        lp = torch.log_softmax(logits, -1)
        mask = (lab != -100); gt = lab.clamp(min=0)
        tok = lp.gather(2, gt.unsqueeze(-1)).squeeze(-1)
        top2 = lp.topk(2, dim=-1).values
        ent = -(lp.exp() * lp).sum(-1)
        rank = (lp > tok.unsqueeze(-1)).sum(-1).float()
        m = mask.float(); n = m.sum(1).clamp(min=1)
        mean_lp = (tok * m).sum(1) / n
        f = torch.stack([
            mean_lp,
            tok.masked_fill(~mask, 1e9).min(1).values,
            tok.masked_fill(~mask, -1e9).max(1).values,
            torch.sqrt((((tok - mean_lp[:, None]) ** 2) * m).sum(1) / n),
            (tok * m).sum(1),
            (ent * m).sum(1) / n,
            ((top2[..., 0] - top2[..., 1]) * m).sum(1) / n,
            (tok.exp() * m).sum(1) / n,
            ((rank == 0).float() * m).sum(1) / n,
            (torch.log1p(rank) * m).sum(1) / n,
            n,
            tok.gather(1, (mask.sum(1) - 1).clamp(min=0)[:, None]).squeeze(1),
        ], dim=1)
        feats.append(f.cpu().numpy())
    return np.nan_to_num(np.concatenate(feats), nan=0.0, posinf=0.0, neginf=0.0)


# ══════════════════════════════════════════════════════════════════════════════
# CLUSTER BOOTSTRAP OVER WRITERS
# ══════════════════════════════════════════════════════════════════════════════
def _resample_writers(groups, rng):
    """Resample writer ids with replacement, return row indices of the
    resampled panel. Duplicated writers contribute their rows twice."""
    uniq = np.unique(groups)
    picked = rng.choice(uniq, size=len(uniq), replace=True)
    idx = []
    lut = {w: np.flatnonzero(groups == w) for w in uniq}
    for w in picked:
        idx.append(lut[w])
    return np.concatenate(idx)


def boot_auc(y, scores, groups, n_boot=2000, seed=S.SEED, alpha=0.05):
    """AUC with a writer-level (cluster) bootstrap CI. Members and non-members
    are resampled independently so class balance is preserved."""
    y = np.asarray(y); scores = np.asarray(scores); groups = np.asarray(groups)
    rng = np.random.RandomState(seed)
    pos, neg = y == 1, y == 0
    vals = []
    for _ in range(n_boot):
        ip = _resample_writers(groups[pos], rng)
        ineg = _resample_writers(groups[neg], rng)
        yy = np.r_[np.ones(len(ip)), np.zeros(len(ineg))]
        ss = np.r_[scores[pos][ip], scores[neg][ineg]]
        if len(np.unique(yy)) < 2:
            continue
        vals.append(roc_auc_score(yy, ss))
    vals = np.array(vals)
    return {"point": roc_auc_score(y, scores),
            "lo": float(np.percentile(vals, 100 * alpha / 2)),
            "hi": float(np.percentile(vals, 100 * (1 - alpha / 2))),
            "boot": vals}


def boot_rate(scores, groups, thr, n_boot=2000, seed=S.SEED, alpha=0.05):
    """Percentage of items above thr, with a writer-level bootstrap CI."""
    scores = np.asarray(scores); groups = np.asarray(groups)
    rng = np.random.RandomState(seed)
    vals = [100.0 * (scores[_resample_writers(groups, rng)] > thr).mean()
            for _ in range(n_boot)]
    vals = np.array(vals)
    return {"point": 100.0 * float((scores > thr).mean()),
            "lo": float(np.percentile(vals, 100 * alpha / 2)),
            "hi": float(np.percentile(vals, 100 * (1 - alpha / 2)))}


def boot_mean(values, groups, n_boot=2000, seed=S.SEED, alpha=0.05):
    values = np.asarray(values, dtype=float); groups = np.asarray(groups)
    rng = np.random.RandomState(seed)
    vals = [values[_resample_writers(groups, rng)].mean() for _ in range(n_boot)]
    vals = np.array(vals)
    return {"point": float(values.mean()),
            "lo": float(np.percentile(vals, 100 * alpha / 2)),
            "hi": float(np.percentile(vals, 100 * (1 - alpha / 2)))}


def thr_at_fpr(nonmember_scores, target_fpr=0.10):
    """Fixed-FPR operating point. Prefer this to Youden's J: with 30 non-member
    writers and near-separable score distributions, J lands in an empty gap and
    the reported percentage swings by whole writers between arms for no reason."""
    return float(np.quantile(np.asarray(nonmember_scores), 1.0 - target_fpr))


def fmt_ci(d, pct=False, prec=3):
    u = "%" if pct else ""
    return f"{d['point']:.{prec}f}{u} [{d['lo']:.{prec}f}, {d['hi']:.{prec}f}]"


# ══════════════════════════════════════════════════════════════════════════════
# CHECKPOINT DISCOVERY
# ══════════════════════════════════════════════════════════════════════════════
def discover_checkpoints(patterns=("unlearned_*.pt", "oracle*.pt")):
    reg = {}
    for pat in patterns:
        hits = sorted(set(glob.glob(f"/kaggle/input/**/{pat}", recursive=True)
                          + glob.glob(f"/kaggle/working/**/{pat}", recursive=True)))
        for p in hits:
            stem = os.path.basename(p)[:-3]
            stem = stem[len("unlearned_"):] if stem.startswith("unlearned_") else stem
            if "_step" in stem:
                tag, step = stem.rsplit("_step", 1)
                reg[f"{tag}@{step}"] = p
            else:
                reg[stem] = p
    return reg


def setup(cfg=S, verbose=True):
    """Load vocab, data, split, feature extractor and the baseline state dict.
    Returns a ctx dict every experiment consumes."""
    if verbose:
        print("=" * 74); print(f"  SETUP  device={DEVICE}"); print("-" * 74)
    vocab = GraphemeVocab.load(os.path.join(cfg.CKPT_ROOT, cfg.VOCAB_FILE))
    base_ck = torch.load(os.path.join(cfg.CKPT_ROOT, cfg.CKPT_FILE),
                         map_location="cpu", weights_only=False)
    w2i = {str(k): int(v) for k, v in base_ck["w2i"].items()}
    df = load_with_writers(cfg)
    retain_df, forget_df, val_df, test_df, w2i = reload_split(df, cfg, w2i=w2i,
                                                              verbose=verbose)
    fe = get_feature_extractor(cfg)
    if verbose:
        print(f"  vocab_size={vocab.vocab_size} | writer classes={len(w2i)}")
    return {"vocab": vocab, "fe": fe, "df": df, "w2i": w2i,
            "retain_df": retain_df, "forget_df": forget_df,
            "val_df": val_df, "test_df": test_df,
            "base_sd": base_ck["model"], "n_classes": len(w2i), "cfg": cfg}


print("pathok_common loaded | device:", DEVICE)

In [ ]:
"""
exp0_diagnose_baseline.py
=========================
~3 min. RUN THIS BEFORE ANYTHING ELSE.

THE SYMPTOM
-----------
The baseline reports 48.25% writer accuracy on its own training members, when
the original retrain logged 100.00%. A 1-epoch oracle beats it by 40 points.
Nothing downstream is interpretable until this is explained.

THE DECISION TREE THIS RUNS
---------------------------
  A. Does the checkpoint's own stored gate dict say 100%?
       yes -> the checkpoint is fine, the CURRENT pipeline is broken. Go to C-F.
       no  -> the checkpoint is not the one you think it is. Go to B.
  B. Key-level load report, un-suppressed, plus writer-head weight statistics.
     A head that was reinitialised or shape-mismatched shows up here.
  C. Are the split sizes 38,275 / 7,739 / 2,015 / 18,067? If not, the dataframe
     changed (glob order, cache, or the label dedupe) and the split is not the
     frozen one.
  D. Are any wid values -1 in retain/forget? A silent map failure sends every
     row to -1 and accuracy collapses without an error.
  E. PER-WRITER accuracy histogram. THIS IS THE KEY TEST:
       bimodal (writers at ~100% and ~0%) -> index/permutation problem on a
         subset of classes
       uniformly degraded (most writers 30-70%) -> input pipeline drift, i.e.
         the images the model sees now differ from the ones it trained on
  F. Confusion offset: is predicted_wid - true_wid a constant, or does it map
     through a consistent permutation? Reveals an off-by-k or a sorted() vs
     checkpoint w2i mismatch.
  G. Fast vs slow image processor, same batch, same weights. The transformers
     default flipped at some point; if this moves the number, that is the cause.

FREE RESULT WHILE YOU ARE HERE
------------------------------
Baseline forget CER, paired with the oracle's forget CER of 15.13%. Same 20
writers, same images, membership as the only manipulated variable. That
difference is the cleanest causal estimate of memorisation in the project and
it costs two extra minutes.
"""

import os
import json
import numpy as np
import pandas as pd
import torch
from collections import Counter
from torch.utils.data import DataLoader
from transformers import AutoImageProcessor


EXPECTED = {"retain": 38275, "forget": 7739, "val": 2015, "test": 18067,
            "total": 66096, "writers": 125, "classes": 95}


def hr(t=""):
    print("\n" + "=" * 78); print(f"  {t}"); print("-" * 78)


def main(cfg=S):
    hr("A. CHECKPOINT SELF-REPORT")
    p = os.path.join(cfg.CKPT_ROOT, cfg.CKPT_FILE)
    print(f"  file: {p}")
    print(f"  size: {os.path.getsize(p)/1e6:.1f} MB")
    ck = torch.load(p, map_location="cpu", weights_only=False)
    print(f"  top-level keys: {list(ck.keys())}")
    if "gate" in ck:
        print("  STORED GATE FROM THE ORIGINAL RUN:")
        print(json.dumps(ck["gate"], indent=4, default=float))
        print("  --> If this says retain/forget ACC ~100 and you now measure 48,")
        print("      the checkpoint is fine and the CURRENT pipeline is at fault.")
    else:
        print("  [!] no 'gate' key -- this may not be baseline_clean_final.pt")

    hr("B. LOAD REPORT (un-suppressed) + WRITER HEAD STATISTICS")
    sd = ck["model"]
    w2i = {str(k): int(v) for k, v in ck["w2i"].items()}
    vocab = GraphemeVocab.load(os.path.join(cfg.CKPT_ROOT, cfg.VOCAB_FILE))
    model = build_model(vocab, len(w2i))
    res = model.load_state_dict(sd, strict=False)
    miss = [k for k in res.missing_keys]
    unexp = list(res.unexpected_keys)
    print(f"  missing    ({len(miss)}): {miss[:10]}")
    print(f"  unexpected ({len(unexp)}): {unexp[:10]}")
    crit = [k for k in miss if not k.startswith("ved.encoder.pooler")]
    print(f"  CRITICAL missing: {len(crit)} {crit[:10]}")

    W = sd.get("writer_head.weight"); B = sd.get("writer_head.bias")
    if W is None:
        print("  [!!] writer_head.weight ABSENT from the checkpoint -- the head is "
              "randomly initialised and 48% would be impossible; something else "
              "is going on")
    else:
        rn = W.float().norm(dim=1)
        print(f"  writer_head.weight  shape {tuple(W.shape)}  "
              f"|W| mean {W.float().abs().mean():.4f}  std {W.float().std():.4f}")
        print(f"  per-class row norms: min {rn.min():.3f}  med {rn.median():.3f}  "
              f"max {rn.max():.3f}  (a trained head has tight, similar norms; a "
              f"reinitialised one is near-uniform and small)")
        print(f"  bias: mean {B.float().mean():.4f}  std {B.float().std():.4f}")
        dead = int((rn < 0.1 * rn.median()).sum())
        print(f"  near-zero rows (possible never-supervised classes): {dead}")

    model = model.to(DEVICE).eval()

    hr("C. DATA + SPLIT RECONSTRUCTION")
    df = load_with_writers(cfg, use_cache=False)   # bypass cache deliberately
    print(f"  total {len(df):,} (expected {EXPECTED['total']:,})  "
          f"writers {df['writer'].nunique()} (expected {EXPECTED['writers']})")
    retain_df, forget_df, val_df, test_df, w2i2 = reload_split(df, cfg, w2i=w2i)
    got = {"retain": len(retain_df), "forget": len(forget_df),
           "val": len(val_df), "test": len(test_df)}
    for k, v in got.items():
        flag = "OK " if v == EXPECTED[k] else "!! "
        print(f"  {flag}{k:<7} {v:>7,}  expected {EXPECTED[k]:>7,}")
    if any(got[k] != EXPECTED[k] for k in got):
        print("  --> The dataframe is not the one the split was frozen against.")
        print("      Delete /kaggle/working/df_cache.parquet and check the label "
              "dedupe and the glob order.")

    hr("D. wid MAPPING")
    for nm, d in (("retain", retain_df), ("forget", forget_df), ("val", val_df)):
        bad = int((d["wid"] < 0).sum())
        print(f"  {nm:<7} rows {len(d):>7,} | wid==-1: {bad:>6,} "
              f"| unique wid {d['wid'].nunique():>3} | range "
              f"[{d['wid'].min()}, {d['wid'].max()}]")
        if bad:
            print(f"    [!!] {bad} rows failed the writer->index map. Check dtype: "
                  f"df['writer'] is {d['writer'].dtype}, w2i keys are str.")

    built = {w: i for i, w in enumerate(sorted(set(w2i)))}
    print(f"  checkpoint w2i == rebuilt sorted(): {w2i == built}")

    hr("E. PER-WRITER ACCURACY  (the key test)")
    fe = AutoImageProcessor.from_pretrained(cfg.TROCR_MODEL)
    per = {}
    for nm, d in (("retain", retain_df), ("forget", forget_df)):
        sub = d.sample(min(3000, len(d)), random_state=cfg.SEED).reset_index(drop=True)
        loader = DataLoader(WDataset(sub, vocab, fe, cfg), batch_size=cfg.BATCH,
                            shuffle=False, num_workers=cfg.NUM_WORKERS)
        hits, preds, trues = [], [], []
        with torch.no_grad():
            for b in loader:
                pv = b["pixel_values"].to(DEVICE); wid = b["wid"].to(DEVICE)
                enc = model.ved.encoder(pixel_values=pv, return_dict=True)
                wl = model.writer_head(enc.last_hidden_state[:, 0, :])
                pr = wl.argmax(-1)
                hits.append((pr == wid).float().cpu())
                preds.append(pr.cpu()); trues.append(wid.cpu())
        hits = torch.cat(hits).numpy()
        preds = torch.cat(preds).numpy(); trues = torch.cat(trues).numpy()
        acc = 100 * hits.mean()
        t = pd.DataFrame({"writer": sub["writer"].values, "hit": hits})
        pw = t.groupby("writer")["hit"].mean() * 100
        per[nm] = (acc, pw, preds, trues)
        print(f"\n  [{nm}] overall {acc:.2f} %  over {len(pw)} writers")
        print(f"    per-writer acc: min {pw.min():.1f}  q25 {pw.quantile(.25):.1f}  "
              f"med {pw.median():.1f}  q75 {pw.quantile(.75):.1f}  max {pw.max():.1f}")
        hi = int((pw > 90).sum()); lo = int((pw < 10).sum())
        print(f"    writers >90%: {hi}   writers <10%: {lo}   in between: "
              f"{len(pw)-hi-lo}")
        if hi and lo and (hi + lo) > 0.7 * len(pw):
            print("    ==> BIMODAL. A subset of classes is systematically wrong.")
            print("        Cause is an index/permutation problem, not the images.")
            print(f"        broken writers: {sorted(pw[pw<10].index)[:15]}")
        elif pw.between(20, 80).mean() > 0.5:
            print("    ==> UNIFORMLY DEGRADED. Every writer is partly wrong.")
            print("        Cause is the input pipeline: the images the model sees "
                  "now differ from the ones it trained on. Go to section G.")
        else:
            print("    ==> Looks healthy.")

    hr("F. CONFUSION STRUCTURE (retain)")
    _, _, preds, trues = per["retain"]
    wrong = preds != trues
    if wrong.any():
        off = (preds[wrong] - trues[wrong])
        c = Counter(off.tolist()).most_common(5)
        print(f"  errors {wrong.sum():,}/{len(preds):,}")
        print(f"  most common (pred - true) offsets: {c}")
        if c and c[0][1] > 0.3 * wrong.sum():
            print(f"  ==> A single dominant offset of {c[0][0]} means the labels "
                  f"are shifted, not the model.")
        i2w = {v: k for k, v in w2i.items()}
        mm = Counter((i2w.get(int(t), t), i2w.get(int(p), p))
                     for t, p in zip(trues[wrong], preds[wrong])).most_common(8)
        print(f"  most common (true_writer -> pred_writer): {mm}")
    else:
        print("  no errors")

    hr("G. FAST vs SLOW IMAGE PROCESSOR")
    sub = retain_df.sample(400, random_state=cfg.SEED).reset_index(drop=True)
    for use_fast in (True, False):
        try:
            fe2 = AutoImageProcessor.from_pretrained(cfg.TROCR_MODEL,
                                                     use_fast=use_fast)
        except Exception as e:
            print(f"  use_fast={use_fast}: unavailable ({e})"); continue
        loader = DataLoader(WDataset(sub, vocab, fe2, cfg), batch_size=cfg.BATCH,
                            shuffle=False, num_workers=cfg.NUM_WORKERS)
        c = t = 0
        with torch.no_grad():
            for b in loader:
                pv = b["pixel_values"].to(DEVICE); wid = b["wid"].to(DEVICE)
                enc = model.ved.encoder(pixel_values=pv, return_dict=True)
                wl = model.writer_head(enc.last_hidden_state[:, 0, :])
                c += (wl.argmax(-1) == wid).sum().item(); t += len(wid)
        print(f"  use_fast={str(use_fast):<5}  retain wACC {100*c/max(t,1):6.2f} %")
    print("  ==> If these differ by more than a point, the processor default "
          "changed between your retrain and now, and that is the whole story.")

    hr("H. BASELINE RECOGNITION -- including the forget CER you need")
    out = {}
    for nm, d in (("retain", retain_df), ("forget", forget_df),
                  ("val", val_df), ("test", test_df)):
        q = rec_quality(model, d, vocab, fe, cfg, n=1000)
        out[nm] = q
        print(f"  [{nm:<6}] WER {q['WER']:6.2f} %  CER {q['CER']:5.2f} %  "
              f"runaway {q['runaway']}")
    print("\n  MEMORISATION ESTIMATE (fill in the oracle numbers from exp1):")
    print(f"    baseline forget CER : {out['forget']['CER']:.2f} %   (members)")
    print(f"    oracle   forget CER : 15.13 %   (same writers, non-members)")
    print(f"    difference          : {15.13 - out['forget']['CER']:.2f} points")
    print("    Same 20 writers, same images, membership as the only manipulated")
    print("    variable. This is the cleanest causal memorisation estimate you")
    print("    have -- much stronger than the retain-vs-test contrast, which")
    print("    confounds membership with writer identity.")

    return {"gate": ck.get("gate"), "per_writer": {k: v[1] for k, v in per.items()},
            "recognition": out}


if __name__ == "__main__":
    main()

In [ ]:
"""
exp2b_probe_repair.py
=====================
~30-50 min. Replaces exp2_reprobe.py.

WHY THE FIRST VERSION HAD NO RESOLUTION
---------------------------------------
Oracle floor 82.89%, baseline ceiling 90.40%, CIs 10-20 points wide. A model
that never saw those 20 writers separates them almost as well as the model that
trained on them. The '% of range' metric was uninterpretable, and so is any
claim built on asymptotic closed-set accuracy.

Three causes, three fixes:

 1. THE TASK WAS TOO EASY. 250 words per writer, 20 classes, 768-d features.
    Fix: probe the SAMPLE-EFFICIENCY curve. Memorisation, if present, shows as
    the model needing fewer examples to place an unseen page of a known writer,
    not as a higher asymptote. Differences live at 1-5 shots, which the original
    design skipped entirely.

 2. CLOSED-SET ACCURACY SATURATES. At 90% there is nowhere to go.
    Fix: VERIFICATION. Cosine similarity over same-writer vs different-writer
    pairs from held-out pages, scored by EER and AUC. This is the standard
    biometric metric and it keeps resolving well past the point where
    identification accuracy flattens.

 3. IT PROBED THE WRONG MODULE. Your own MIA gets per-writer AUC 0.956 from
    DECODER confidence. So memorisation exists and is detectable -- just not in
    the encoder's writer-discriminative geometry, which is dominated by generic
    Bangla style extraction. Fix: probe decoder states too, so the negative
    encoder result is a contrast rather than a dead end.

 4. PER-ARM MAX OVER 20 CONFIGS is a selection artefact. Fix: one fixed config,
    declared up front. Everything else goes in the appendix.

A FOURTH REFERENCE POINT
------------------------
TEST writers under the BASELINE. These 30 were seen by nobody, so their
separability is pure generic style capacity. If forget-under-oracle equals
test-under-baseline, you have measured the encoder's ability to separate any 20
Bangla writers and confirmed that the probe is blind to membership. That is the
publishable claim, and it needs this arm to be made cleanly.

OUTPUT
------
  /kaggle/working/reprobe2/fewshot_curves.csv
  /kaggle/working/reprobe2/verification.csv
  /kaggle/working/reprobe2/summary.csv
"""

import os
import itertools
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, roc_curve
from tqdm.auto import tqdm

class P:
    OUT = "/kaggle/working/reprobe2"
    ORACLE = "/kaggle/working/oracle/oracle_final.pt"

    WORDS_PER_WRITER = 300
    SHOTS = [1, 2, 5, 10, 25, 50, 150]     # probe-training words per writer
    N_REPEATS = 5                          # resamples of the k-shot support set
    LAYER = 12                             # fixed. declared before looking.
    POOL = "cls"                           # fixed. 'meanpatch' reported alongside.
    ALSO_POOL = "meanpatch"
    PROBE_DECODER = True                   # the contrast that matters

    N_PAIRS = 40000                        # verification pairs per arm/group
    N_RETAIN_CONTROL = 20

    ARMS = ["baseline", "oracle", "wic_unmasked@300", "wic_unmasked@800",
            "random_label@800", "fisher@800", "amnesiac@800", "delete@800"]
    GROUPS = ["forget", "retain_control", "test_reference"]


os.makedirs(P.OUT, exist_ok=True)


# ──────────────────────────────────────────────────────────────────────────────
@torch.no_grad()
def embed(model, df, vocab, fe, cfg=S):
    """Encoder CLS + mean-patch at a FIXED layer, plus a decoder-state pooling.
    The decoder vector is the mean hidden state over the teacher-forced label
    positions -- the same computation the 12-dim MIA features summarise, so an
    encoder/decoder contrast is apples to apples."""
    loader = DataLoader(WDataset(df, vocab, fe, cfg), batch_size=cfg.BATCH,
                        shuffle=False, num_workers=cfg.NUM_WORKERS)
    model.eval()
    out = {"cls": [], "meanpatch": [], "decoder": []}
    for b in tqdm(loader, desc="embed", leave=False, dynamic_ncols=True):
        pv = b["pixel_values"].to(DEVICE); lab = b["labels"].to(DEVICE)
        enc = model.ved.encoder(pixel_values=pv, output_hidden_states=True,
                                return_dict=True)
        h = enc.hidden_states[P.LAYER].float()
        out["cls"].append(h[:, 0, :].cpu().numpy())
        out["meanpatch"].append(h[:, 1:, :].mean(1).cpu().numpy())
        if P.PROBE_DECODER:
            d = model.ved(encoder_outputs=enc, labels=lab,
                          output_hidden_states=True, return_dict=True)
            dh = d.decoder_hidden_states[-1].float()
            m = (lab != -100).float().unsqueeze(-1)
            out["decoder"].append(((dh * m).sum(1) /
                                   m.sum(1).clamp(min=1)).cpu().numpy())
    return ({k: np.concatenate(v) for k, v in out.items() if v},
            df["writer"].values, df["page"].values)


def page_partition(pages, writers, seed=S.SEED):
    """Held-out pages per writer. Support sets are always drawn from the train
    pages, evaluation always from the held-out pages."""
    rng = np.random.RandomState(seed)
    te = np.zeros(len(pages), dtype=bool)
    for w in np.unique(writers):
        m = writers == w
        wp = np.unique(pages[m]); rng.shuffle(wp)
        k = max(1, min(len(wp) - 1, int(round(0.3 * len(wp))))) if len(wp) > 1 else 0
        if k:
            te |= np.isin(pages, wp[:k]) & m
        else:
            idx = np.flatnonzero(m); te[idx[: max(1, len(idx) // 3)]] = True
    return ~te, te


def fewshot_curve(X, writers, tr, te, arm, group, feat):
    """Accuracy vs number of support words per writer, averaged over resamples.
    Sample efficiency, not asymptote, is where memorisation would show.

    FIX (2026-08-13): the original version aborted the ENTIRE curve the moment
    ANY one class had an empty train-page pool (page_partition() puts a writer
    with too few total samples entirely into the test split). One under-sampled
    writer was silently zeroing out every arm/group/feat result. Now that writer
    is dropped from the support set only -- it still sits in the held-out eval
    set and is simply never predicted correctly, which is the honest outcome for
    an attacker with zero training shots on it."""
    y, classes = pd.factorize(writers)
    all_classes = np.unique(y)
    train_classes = [c for c in all_classes if (tr & (y == c)).any()]
    no_train = [classes[c] for c in all_classes if c not in train_classes]
    if no_train:
        print(f"    [{arm}|{group}|{feat}] writer(s) with zero train-page rows "
              f"(too few samples to split) -- excluded from the support set, "
              f"still scored as misses in eval: {no_train}")
    rows = []
    for shots in P.SHOTS:
        accs, hits_all, w_all = [], [], []
        for rep in range(P.N_REPEATS):
            rng = np.random.RandomState(S.SEED + rep)
            sel = []
            for c in train_classes:
                pool = np.flatnonzero(tr & (y == c))
                sel.append(rng.choice(pool, min(shots, len(pool)), replace=False))
            if not sel:
                continue
            sel = np.concatenate(sel)
            if len(np.unique(y[sel])) < 2:
                continue
            clf = make_pipeline(StandardScaler(),
                                LogisticRegression(max_iter=3000, C=1.0))
            clf.fit(X[sel], y[sel])
            pred = clf.predict(X[te])
            hit = (pred == y[te]).astype(float)
            accs.append(100 * hit.mean())
            hits_all.append(hit); w_all.append(writers[te])
        if not accs:
            continue
        ci = boot_mean(np.concatenate(hits_all), np.concatenate(w_all), n_boot=800)
        rows.append({"arm": arm, "group": group, "feat": feat, "shots": shots,
                     "acc": 100 * ci["point"], "lo": 100 * ci["lo"],
                     "hi": 100 * ci["hi"], "acc_sd_over_reps": float(np.std(accs)),
                     "chance": 100 / len(np.unique(y)), "n_eval": int(te.sum())})
    return rows


def verification(X, writers, pages, te, arm, group, feat, n_pairs=P.N_PAIRS):
    """Same-writer vs different-writer cosine similarity over held-out pages.
    Pairs are always ACROSS pages, so page texture cannot carry the decision."""
    idx = np.flatnonzero(te)
    if len(idx) < 50:
        return None
    Z = X[idx]
    Z = Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-8)
    w = writers[idx]; pg = pages[idx]
    rng = np.random.RandomState(S.SEED)
    a = rng.randint(0, len(idx), n_pairs)
    b = rng.randint(0, len(idx), n_pairs)
    keep = (a != b) & (pg[a] != pg[b])          # never within-page
    a, b = a[keep], b[keep]
    if len(a) < 200:
        return None
    sim = (Z[a] * Z[b]).sum(1)
    lab = (w[a] == w[b]).astype(int)
    if lab.sum() < 20 or (1 - lab).sum() < 20:
        return None
    auc = roc_auc_score(lab, sim)
    fpr, tpr, _ = roc_curve(lab, sim)
    eer = float(fpr[np.nanargmin(np.abs(fpr - (1 - tpr)))])
    ci = boot_auc(lab, sim, w[a], n_boot=600)
    return {"arm": arm, "group": group, "feat": feat, "ver_auc": auc,
            "ver_auc_lo": ci["lo"], "ver_auc_hi": ci["hi"], "eer": 100 * eer,
            "n_pos": int(lab.sum()), "n_neg": int((1 - lab).sum())}


# ──────────────────────────────────────────────────────────────────────────────
def main():
    ctx = setup()
    vocab, fe = ctx["vocab"], ctx["fe"]
    reg = discover_checkpoints()
    if os.path.exists(P.ORACLE):
        reg["oracle"] = P.ORACLE
    reg.setdefault("oracle", reg.get("oracle_final"))
    print(f"\n  checkpoints on disk: {sorted(k for k in reg if reg.get(k))}")
    missing = [a for a in P.ARMS if a not in ("baseline",) and not reg.get(a)]
    if missing:
        print(f"  [!] not found, will be skipped: {missing}")
        print("      Attach the Stage II notebook output before quoting anything.")

    rng = np.random.RandomState(S.SEED)
    rw = rng.choice(sorted(ctx["retain_df"]["writer"].unique()),
                    P.N_RETAIN_CONTROL, replace=False)
    tw = rng.choice(sorted(ctx["test_df"]["writer"].unique()),
                    P.N_RETAIN_CONTROL, replace=False)
    sets = {
        "forget": sample_per_writer(ctx["forget_df"], P.WORDS_PER_WRITER),
        "retain_control": sample_per_writer(
            ctx["retain_df"][ctx["retain_df"]["writer"].isin(rw)]
            .reset_index(drop=True), P.WORDS_PER_WRITER),
        "test_reference": sample_per_writer(
            ctx["test_df"][ctx["test_df"]["writer"].isin(tw)]
            .reset_index(drop=True), P.WORDS_PER_WRITER),
    }
    for k, v in sets.items():
        print(f"  {k:<15} {len(v):>6,} words / {v['writer'].nunique()} writers "
              f"(chance {100/v['writer'].nunique():.2f}%)")

    model = build_model(vocab, ctx["n_classes"])
    curve_rows, ver_rows = [], []
    for arm in P.ARMS:
        if arm == "baseline":
            sd = ctx["base_sd"]
        else:
            p = reg.get(arm)
            if not p or not os.path.exists(p):
                continue
            sd = torch.load(p, map_location="cpu", weights_only=False)["model"]
        load_into(model, sd, arm, strict_report=False)
        head = writer_acc(model, ctx["forget_df"], vocab, fe, n=800)
        print(f"\n  --- {arm} ---  head forget wACC {head:.2f} %")

        for group, d in sets.items():
            E, w, pg = embed(model, d, vocab, fe)
            tr, te = page_partition(pg, w)
            for feat, X in E.items():
                if feat == P.ALSO_POOL and group != "forget":
                    continue                    # keep the grid small
                curve_rows += [dict(r, head_forget_wACC=head)
                               for r in fewshot_curve(X, w, tr, te, arm, group, feat)]
                v = verification(X, w, pg, te, arm, group, feat)
                if v:
                    ver_rows.append(dict(v, head_forget_wACC=head))

    C = pd.DataFrame(curve_rows); V = pd.DataFrame(ver_rows)
    C.to_csv(f"{P.OUT}/fewshot_curves.csv", index=False)
    V.to_csv(f"{P.OUT}/verification.csv", index=False)
    if C.empty:
        print("\n  [!!] fewshot_curves is empty -- no arm/group/feat combo "
              "produced a usable curve. Identification tables below will be "
              "blank; verification.csv may still have data.")
        C = pd.DataFrame(columns=["arm", "group", "feat", "shots", "acc", "lo",
                                   "hi", "acc_sd_over_reps", "chance", "n_eval",
                                   "head_forget_wACC"])

    # ── SAMPLE EFFICIENCY TABLE (the point of the rewrite) ────────────────────
    print("\n" + "=" * 92)
    print(f"  FEW-SHOT IDENTIFICATION, forget writers, {P.POOL} L{P.LAYER}")
    print("  Differences, if any, live at 1-5 shots. The asymptote is generic")
    print("  Bangla style capacity and tells you nothing about membership.")
    print("-" * 92)
    q = C[(C.group == "forget") & (C.feat == P.POOL)]
    if len(q):
        piv = q.pivot_table(index="arm", columns="shots", values="acc")
        print(piv.round(1).to_string())
    print("\n  Same table, DECODER states (where your MIA finds signal):")
    qd = C[(C.group == "forget") & (C.feat == "decoder")]
    if len(qd):
        print(qd.pivot_table(index="arm", columns="shots", values="acc")
                .round(1).to_string())

    print("\n" + "=" * 92)
    print("  GENERIC-CAPACITY CONTROL")
    print("-" * 92)
    ref = C[(C.feat == P.POOL) & (C.shots == max(P.SHOTS))]
    for _, r in ref.iterrows():
        print(f"  {r['arm']:<20} {r['group']:<16} {r['acc']:6.2f}% "
              f"[{r['lo']:5.1f}, {r['hi']:5.1f}]  (chance {r['chance']:.1f}%)")
    print("\n  If forget-under-oracle ~= test_reference-under-baseline, the probe")
    print("  is measuring how well a Bangla-trained ViT separates ANY 20 writers,")
    print("  not whether these 20 were deleted. Say that explicitly -- it is the")
    print("  result, and it invalidates every published re-probe audit that was")
    print("  reported without an oracle floor.")

    print("\n" + "=" * 92)
    print("  VERIFICATION (same-writer vs different-writer, across pages)")
    print("-" * 92)
    if len(V):
        print(f"  {'arm':<20} {'group':<16} {'feat':<11} {'AUC':>18} {'EER':>7}")
        for _, r in V.sort_values(["feat", "group", "arm"]).iterrows():
            print(f"  {r['arm']:<20} {r['group']:<16} {r['feat']:<11} "
                  f"{r['ver_auc']:.3f}[{r['ver_auc_lo']:.2f},{r['ver_auc_hi']:.2f}] "
                  f"{r['eer']:6.2f}%")
    print("\n  EER keeps resolving where closed-set accuracy has saturated. If the")
    print("  arms separate here but not in the identification table, report EER as")
    print("  the primary probe metric and say why.")

    summ = (C[C.shots == min(P.SHOTS)]
            .merge(C[C.shots == max(P.SHOTS)], on=["arm", "group", "feat"],
                   suffixes=("_lowshot", "_highshot")))
    summ.to_csv(f"{P.OUT}/summary.csv", index=False)
    print(f"\n  saved -> {P.OUT}/fewshot_curves.csv, verification.csv, summary.csv")
    return C, V


if __name__ == "__main__":
    main()

In [ ]:
"""
exp3b_relearn_repair.py
=======================
~2 h. Replaces exp3_relearn.py.

WHAT WAS BROKEN
---------------
The relearn objective had a recognition term but NO retain writer term, so
fine-tuning on 20 classes demolished the 95-way head. Oracle retain wACC went
88.75% -> 5.00% in 200 steps. The reported "64.62% forget accuracy" was mostly
20-way accuracy on a wrecked model, not recovery of anything, and the summary
metrics degenerated with it (recovered_frac 277%, steps_to_half 0).

FOUR REPAIRS
------------
 1. RETAIN REPLAY. Every step interleaves a retain batch with writer CE, so the
    95-way head has to stay intact while the attacker pushes on 20 classes.
    Recovery now has to happen WITHOUT collateral, which is both the realistic
    threat model and the only way the number means anything. Retain wACC is a
    hard gate: any step where it drops below RETAIN_FLOOR invalidates the row,
    and the script says so instead of quietly reporting the curve.

 2. TWO EVALUATION MODES, both reported.
      full95     : argmax over all 95 logits -- comparable to every other
                   number in the paper.
      restrict20 : argmax over the 20 forget logits only -- isolates "can the
                   model tell these writers apart" from "is the head healthy".
    Divergence between the two is itself diagnostic of head damage.

 3. SMALLER BUDGETS, DENSER EARLY EVALS. A model that never saw the forget
    writers reached 65% from 5 words each in 200 steps, so the discriminative
    region is k in {1,2,3} and steps < 50. The original grid measured only the
    saturated tail.

 4. HONEST METRICS.
      - The recovery target is the ORIGINAL BASELINE forget wACC, and the
        baseline arm is excluded from ratios (it starts at the target, so every
        ratio involving it is 0.00x or 1.00x by construction).
      - Primary statistic is EXCESS OVER ORACLE AT MATCHED STEPS: the area
        between an arm's curve and the oracle's, integrated over the budget.
        This does not degenerate when either curve saturates.
      - steps_to_target is reported only where it is finite and the arm did not
        start above target.

READ THE RESULT THIS WAY
------------------------
The oracle curve is the from-scratch rate. If it is already steep -- and it is --
then few-shot relearning has limited discriminative power here regardless of the
repairs, for the same reason the encoder probe does: generic style features make
20 Bangla writers easy to separate from almost nothing. Report that as the
finding, with the oracle curve as the evidence, rather than reporting an
unlearned arm's curve on its own.
"""

import os
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader


class R:
    OUT = "/kaggle/working/relearn2"
    ORACLE = "/kaggle/working/oracle/oracle_final.pt"

    K_PER_WRITER = [1, 2, 3, 5]
    STEPS = 150
    EVAL_STEPS = [0, 2, 5, 10, 15, 20, 30, 45, 60, 90, 120, 150]
    MICRO_F = 8               # forget images per step
    MICRO_R = 8               # retain replay images per step
    LR = 1e-5
    WD = 0.0
    CLIP = 1.0
    LAMBDA_F = 1.0            # forget writer CE (the attack)
    LAMBDA_R = 1.0            # retain writer CE (the replay -- this was missing)
    LAMBDA_REC = 0.5
    AMP = True

    EVAL_N = 800
    RETAIN_EVAL_N = 400
    RETAIN_FLOOR = 0.90       # fraction of the arm's own starting retain wACC

    ARMS = ["oracle", "wic_unmasked@300", "wic_unmasked@800",
            "random_label@800", "fisher@800", "amnesiac@800", "delete@800", "baseline"]


os.makedirs(R.OUT, exist_ok=True)


def page_disjoint_kshot(forget_df, k, seed=S.SEED):
    rng = np.random.RandomState(seed)
    tr_idx, te_idx = [], []
    for w, g in forget_df.groupby("writer"):
        pages = g["page"].unique().copy(); rng.shuffle(pages)
        if len(pages) < 2:
            gi = g.sample(frac=1.0, random_state=seed).index.to_numpy()
            tr_idx.append(gi[:k]); te_idx.append(gi[k:]); continue
        n_tr = max(1, len(pages) // 3)
        tr_pages = set(pages[:n_tr])
        gtr = g[g["page"].isin(tr_pages)]
        gte = g[~g["page"].isin(tr_pages)]
        tr_idx.append(gtr.sample(min(k, len(gtr)), random_state=seed).index.to_numpy())
        te_idx.append(gte.index.to_numpy())
    return (forget_df.loc[np.concatenate(tr_idx)].reset_index(drop=True),
            forget_df.loc[np.concatenate(te_idx)].reset_index(drop=True))


def infinite(loader):
    while True:
        for b in loader:
            yield b


@torch.no_grad()
def forget_acc_two_ways(model, df, vocab, fe, forget_idx, cfg=S, n=R.EVAL_N):
    """full95 = argmax over all classes. restrict20 = argmax over forget logits
    only. Divergence between them means the head is damaged, not that identity
    returned."""
    d = df.sample(min(n, len(df)), random_state=cfg.SEED).reset_index(drop=True)
    loader = DataLoader(WDataset(d, vocab, fe, cfg), batch_size=cfg.BATCH,
                        shuffle=False, num_workers=cfg.NUM_WORKERS)
    was = model.training; model.eval()
    f95, f20 = [], []
    fi = forget_idx.to(DEVICE)
    for b in loader:
        pv = b["pixel_values"].to(DEVICE); wid = b["wid"].to(DEVICE)
        enc = model.ved.encoder(pixel_values=pv, return_dict=True)
        wl = model.writer_head(enc.last_hidden_state[:, 0, :]).float()
        f95.append((wl.argmax(-1) == wid).float().cpu())
        sub = wl[:, fi]
        pred = fi[sub.argmax(-1)]
        f20.append((pred == wid).float().cpu())
    if was:
        model.train()
    return (100 * float(torch.cat(f95).mean()), 100 * float(torch.cat(f20).mean()))


def relearn(model, state, ctx, kshot_df, eval_df, forget_idx, tag, cfg=S):
    seed_all(S.SEED)
    model.load_state_dict(state, strict=True)
    vocab, fe = ctx["vocab"], ctx["fe"]

    opt = torch.optim.AdamW(model.parameters(), lr=R.LR, weight_decay=R.WD)
    scaler = torch.amp.GradScaler("cuda", enabled=R.AMP and DEVICE == "cuda")

    bf = min(R.MICRO_F, len(kshot_df))
    f_it = infinite(DataLoader(WDataset(kshot_df, vocab, fe, cfg), batch_size=bf,
                               shuffle=True, drop_last=len(kshot_df) > bf,
                               num_workers=cfg.NUM_WORKERS))
    r_it = infinite(DataLoader(WDataset(ctx["retain_df"], vocab, fe, cfg),
                               batch_size=R.MICRO_R, shuffle=True, drop_last=True,
                               num_workers=cfg.NUM_WORKERS))

    def snap(step):
        a95, a20 = forget_acc_two_ways(model, eval_df, vocab, fe, forget_idx, cfg)
        ra = writer_acc(model, ctx["retain_df"], vocab, fe, cfg, n=R.RETAIN_EVAL_N)
        return {"step": step, "forget_full95": a95, "forget_restrict20": a20,
                "retain_wACC": ra, "tag": tag}

    rows = [snap(0)]
    r0 = rows[0]["retain_wACC"]
    print(f"    step   0  f95 {rows[0]['forget_full95']:6.2f}  "
          f"f20 {rows[0]['forget_restrict20']:6.2f}  retain {r0:6.2f}")

    model.train()
    for step in range(1, R.STEPS + 1):
        b_f, b_r = next(f_it), next(r_it)
        nf = b_f["pixel_values"].shape[0]
        pv = torch.cat([b_f["pixel_values"], b_r["pixel_values"]]).to(DEVICE)
        lab = torch.cat([b_f["labels"], b_r["labels"]]).to(DEVICE)
        wid = torch.cat([b_f["wid"], b_r["wid"]]).to(DEVICE)

        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", dtype=torch.float16,
                                enabled=R.AMP and DEVICE == "cuda"):
            enc = model.ved.encoder(pixel_values=pv, return_dict=True)
            wlog = model.writer_head(enc.last_hidden_state[:, 0, :])
            rec = model.ved(encoder_outputs=enc, labels=lab, return_dict=True)
        wl = wlog.float()
        l_f = F.cross_entropy(wl[:nf], wid[:nf])
        l_r = F.cross_entropy(wl[nf:], wid[nf:])      # THE REPAIR
        loss = R.LAMBDA_F * l_f + R.LAMBDA_R * l_r + R.LAMBDA_REC * rec.loss.float()

        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), R.CLIP)
        scaler.step(opt); scaler.update()

        if step in R.EVAL_STEPS:
            s = snap(step); rows.append(s)
            warn = "  [!] retain collapsed" if s["retain_wACC"] < R.RETAIN_FLOOR * r0 else ""
            print(f"    step {step:3d}  f95 {s['forget_full95']:6.2f}  "
                  f"f20 {s['forget_restrict20']:6.2f}  "
                  f"retain {s['retain_wACC']:6.2f}{warn}")
            model.train()

    model.eval()
    c = pd.DataFrame(rows)
    c["retain_start"] = r0
    c["valid"] = c["retain_wACC"] >= R.RETAIN_FLOOR * r0
    return c


def auc_curve(c, col="forget_full95"):
    """Normalised area under the recovery curve over the step budget. Robust
    where steps_to_target degenerates."""
    x = c["step"].values.astype(float); y = c[col].values.astype(float)
    return float(np.trapz(y, x) / max(x.max() - x.min(), 1e-9))


def main():
    ctx = setup()
    vocab, fe = ctx["vocab"], ctx["fe"]
    reg = discover_checkpoints()
    if os.path.exists(R.ORACLE):
        reg["oracle"] = R.ORACLE
    reg.setdefault("oracle", reg.get("oracle_final"))
    print(f"\n  checkpoints on disk: {sorted(k for k in reg if reg.get(k))}")

    forget_idx = torch.tensor(sorted(ctx["forget_df"]["wid"].unique()),
                              dtype=torch.long)
    print(f"  {len(forget_idx)} forget classes; restrict20 chance = "
          f"{100/len(forget_idx):.2f} %")

    model = build_model(vocab, ctx["n_classes"])
    load_into(model, ctx["base_sd"], "baseline", strict_report=False)
    target = writer_acc(model, ctx["forget_df"], vocab, fe, n=800)
    print(f"  recovery target (original baseline forget wACC): {target:.2f} %")
    if target < 80:
        print("  [!!] The baseline is not at ~100% on its own members. Run "
              "exp0_diagnose_baseline.py first -- this target is not credible "
              "and every ratio below inherits the problem.")

    curves, summ = [], []
    for arm in R.ARMS:
        if arm == "baseline":
            sd = ctx["base_sd"]
        else:
            p = reg.get(arm)
            if not p or not os.path.exists(p):
                print(f"\n  [skip] {arm}: not found")
                continue
            sd = torch.load(p, map_location="cpu", weights_only=False)["model"]
        load_into(model, sd, arm, strict_report=False)
        state = copy.deepcopy(model.state_dict())

        for k in R.K_PER_WRITER:
            tr, te = page_disjoint_kshot(ctx["forget_df"], k)
            print(f"\n  --- {arm} | k={k}/writer ({len(tr)} attack words, "
                  f"{len(te):,} held-out, page-disjoint) ---")
            c = relearn(model, state, ctx, tr, te, forget_idx, f"{arm}|k{k}")
            c["arm"] = arm; c["k"] = k
            curves.append(c)
            summ.append({
                "arm": arm, "k": k,
                "start_f95": c["forget_full95"].iloc[0],
                "final_f95": c["forget_full95"].iloc[-1],
                "final_f20": c["forget_restrict20"].iloc[-1],
                "auc_f95": auc_curve(c, "forget_full95"),
                "retain_start": c["retain_start"].iloc[0],
                "retain_final": c["retain_wACC"].iloc[-1],
                "all_steps_valid": bool(c["valid"].all()),
                "recovered_frac": c["forget_full95"].iloc[-1] / max(target, 1e-9)})

    C = pd.concat(curves, ignore_index=True)
    Sm = pd.DataFrame(summ)
    C.to_csv(f"{R.OUT}/relearn_curves.csv", index=False)

    if "oracle" in set(Sm["arm"]):
        ref = Sm[Sm.arm == "oracle"].set_index("k")
        Sm["excess_auc_vs_oracle"] = Sm.apply(
            lambda r: r["auc_f95"] - ref.loc[r["k"], "auc_f95"], axis=1)
        Sm["excess_final_vs_oracle"] = Sm.apply(
            lambda r: r["final_f95"] - ref.loc[r["k"], "final_f95"], axis=1)
    Sm.to_csv(f"{R.OUT}/relearn_summary.csv", index=False)

    print("\n" + "=" * 104)
    print("  FEW-SHOT RELEARNING, repaired  (retain replay on, restricted-logit "
          "eval reported)")
    print("-" * 104)
    print(f"  {'arm':<20} {'k':>2} {'start':>7} {'final95':>8} {'final20':>8} "
          f"{'AUC':>7} {'ΔAUC vs orc':>12} {'retain':>14} {'valid':>6}")
    for _, r in Sm.iterrows():
        ex = r.get("excess_auc_vs_oracle", np.nan)
        print(f"  {r['arm']:<20} {int(r['k']):>2} {r['start_f95']:6.2f}% "
              f"{r['final_f95']:7.2f}% {r['final_f20']:7.2f}% {r['auc_f95']:7.2f} "
              f"{('-' if not np.isfinite(ex) else f'{ex:+11.2f}'):>12} "
              f"{r['retain_start']:5.1f}->{r['retain_final']:5.1f}  "
              f"{str(r['all_steps_valid']):>6}")
    print("-" * 104)
    print("  valid=False means retain wACC fell below 90% of its start: the run")
    print("  recovered forget identity by damaging the model, and the row should")
    print("  not be quoted.")
    print("  ΔAUC vs oracle ~ 0 means the arm relearns no faster than a model that")
    print("  never saw these writers -- consistent with real deletion, and also")
    print("  consistent with the attack simply having no resolution. Distinguish")
    print("  the two by looking at whether the ORACLE curve is already steep: if")
    print("  it is, this probe cannot audit deletion here, and that is the result.")
    print(f"\n  saved -> {R.OUT}/relearn_curves.csv, {R.OUT}/relearn_summary.csv")
    return C, Sm


if __name__ == "__main__":
    main()